# Validation — Risk Index vs. Actual 2019/20 Fire Severity

This notebook checks whether the pre-fire-season risk index from
`01_bushfire_risk_mapping.ipynb` actually lines up with what happened on
the ground during the 2019–20 Black Summer fires.

The FESM 2019/20 "Data Download Package" from NSW SEED is an **ESRI
Arc/Info Binary Grid raster** (a folder of `.adf` files), not a
shapefile — every pixel is a fire-severity class code
(unburnt/low/moderate/high/extreme). Since our risk index is also a
raster, this notebook compares them **pixel-to-pixel** rather than doing
zonal statistics over polygons — simpler and avoids any
raster-to-vector conversion.

**Prerequisite (do this locally, not in Colab, before running this notebook):**

1. Download and unzip the FESM 2019/20 "Data Download Package" from NSW SEED.
2. Open the FESMv3 metadata PDF that comes with it and note down what each
   pixel class code means (0/1/2/3/4 etc → unburnt/low/moderate/high/extreme).
   Don't assume the mapping — confirm it against the PDF.
3. Run the clipping script to cut the statewide raster down to the Blue
   Mountains AOI:

```bash
pip install rasterio
python src/clip_fire_severity_raster.py \
    --input "D:/download/FireSeverityFESM/fesm_201920" \
    --output data/fesm_2019_20_blue_mountains.tif
```

   This prints the unique pixel values found in the clipped area — cross
   check those against the metadata PDF.

4. Commit `data/fesm_2019_20_blue_mountains.tif` to the repo (should be a
   few hundred KB to a few MB — small enough to commit), then run this
   notebook in Colab.

Pipeline:
1. Rebuild the risk index raster (same as notebook 01) and export it locally
2. Load the clipped fire-severity raster
3. Align both rasters onto the same pixel grid
4. Boxplot: does risk index increase with actual fire severity?
5. Heatmap: predicted risk class vs. observed severity class

## 0. Setup

In [ ]:
import os
if not os.path.exists('nsw-bushfire-risk-mapping'):
    !git clone https://github.com/<your-username>/nsw-bushfire-risk-mapping.git
    %cd nsw-bushfire-risk-mapping

!pip install -q earthengine-api geemap folium rasterio seaborn

In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='your-gee-project-id')

In [ ]:
import sys
sys.path.append('src')

import geemap
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt
import seaborn as sns

from risk_index import mask_s2_clouds, compute_risk_index, DEFAULT_WEIGHTS

## 1. Rebuild the risk index (same AOI/dates as notebook 01)

In [ ]:
aoi = ee.Geometry.Rectangle([150.10, -33.85, 150.45, -33.55])

START_DATE = '2019-10-01'
END_DATE = '2019-11-30'

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    .map(mask_s2_clouds)
)
s2_composite = s2_collection.median().clip(aoi)

dem = ee.Image('USGS/SRTMGL1_003').clip(aoi)

risk_image = compute_risk_index(s2_composite, dem, aoi, weights=DEFAULT_WEIGHTS)
print('Risk image bands:', risk_image.bandNames().getInfo())

## 2. Export the risk raster locally

`geemap.ee_export_image` downloads directly (no Google Cloud Storage
bucket needed) as long as the region is small enough — the Blue Mountains
AOI at 20m resolution is well within that limit.

In [ ]:
RISK_TIF = 'outputs/blue_mountains_risk.tif'

geemap.ee_export_image(
    risk_image.select(['risk_index', 'risk_class']),
    filename=RISK_TIF,
    scale=20,
    region=aoi,
    file_per_band=False,
)

## 3. Load both rasters and align them onto the same grid

The FESM raster and our exported risk raster will almost certainly have
different resolution/grid alignment, so we reproject the fire-severity
raster onto the exact pixel grid of the risk raster using **nearest**
resampling (important: nearest, not bilinear — these are class codes,
not continuous values, so blending them would be meaningless).

In [ ]:
FIRE_TIF = 'data/fesm_2019_20_blue_mountains.tif'

with rasterio.open(RISK_TIF) as risk_src:
    risk_data = risk_src.read()  # bands: [risk_index, risk_class]
    risk_index_arr = risk_data[0]
    risk_class_arr = risk_data[1]
    risk_transform = risk_src.transform
    risk_crs = risk_src.crs
    risk_shape = (risk_src.height, risk_src.width)

with rasterio.open(FIRE_TIF) as fire_src:
    severity_aligned = np.empty(risk_shape, dtype=np.float32)
    reproject(
        source=rasterio.band(fire_src, 1),
        destination=severity_aligned,
        src_transform=fire_src.transform,
        src_crs=fire_src.crs,
        dst_transform=risk_transform,
        dst_crs=risk_crs,
        dst_resolution=None,
        resampling=Resampling.nearest,
    )

print('risk_index shape:', risk_index_arr.shape)
print('severity (aligned) shape:', severity_aligned.shape)
print('unique severity codes present:', np.unique(severity_aligned))

## 4. Map severity codes to labels

**Fill this in from the FESMv3 metadata PDF** — don't guess. The dict
below is a common convention but must be confirmed against your PDF.

In [ ]:
# TODO: confirm these against the FESMv3 metadata PDF before trusting the results
SEVERITY_LABELS = {
    0: 'unburnt',
    1: 'burnt_grassland',
    2: 'low',
    3: 'moderate',
    4: 'high',
    5: 'extreme',
}
NODATA_CODES = [255]  # add whatever nodata value your raster actually uses

## 5. Flatten into a pixel-level DataFrame

In [ ]:
df = pd.DataFrame({
    'risk_index': risk_index_arr.flatten(),
    'risk_class': risk_class_arr.flatten(),
    'severity_code': severity_aligned.flatten(),
})

# Drop nodata / masked pixels
df = df[~df['severity_code'].isin(NODATA_CODES)]
df = df[df['risk_index'].notna() & (df['risk_index'] >= 0)]

df['severity_label'] = df['severity_code'].round().map(SEVERITY_LABELS)
df = df.dropna(subset=['severity_label'])

print(f'{len(df):,} valid pixels for comparison')
df['severity_label'].value_counts()

## 6. Boxplot — risk index distribution by observed fire severity

If the pre-fire-season risk index is doing its job, pixels that actually
burned at **high/extreme** severity should show a **higher** risk-index
distribution than **unburnt/low** severity pixels.

Note: with potentially millions of pixels, plotting every point isn't
useful — we sample down for the boxplot.

In [ ]:
severity_order = ['unburnt', 'low', 'moderate', 'high', 'extreme']
present_order = [s for s in severity_order if s in df['severity_label'].unique()]

sample_df = df.sample(n=min(50_000, len(df)), random_state=42)

plt.figure(figsize=(8, 5))
sns.boxplot(
    data=sample_df, x='severity_label', y='risk_index',
    order=present_order, palette='YlOrRd',
)
plt.xlabel('Observed fire severity (2019/20, FESM)')
plt.ylabel('Pre-fire-season risk index (0-100)')
plt.title('Does pre-season risk index predict actual fire severity?')
plt.tight_layout()
plt.savefig('outputs/validation_boxplot.png', dpi=150)
plt.show()

## 7. Heatmap — predicted risk class vs. observed severity

Normalised by column so each severity class sums to 1 — makes it easy to
read as "of the pixels that actually burned at X severity, what fraction
did we predict as each risk class?". A good pattern to look for: mass
concentrated along the diagonal.

In [ ]:
RISK_CLASS_LABELS = {1: 'Low', 2: 'Moderate', 3: 'High', 4: 'Extreme'}
df['risk_class_label'] = df['risk_class'].round().map(RISK_CLASS_LABELS)

crosstab = pd.crosstab(
    df['risk_class_label'], df['severity_label'], normalize='columns'
)
row_order = [l for l in ['Low', 'Moderate', 'High', 'Extreme'] if l in crosstab.index]
col_order = [s for s in severity_order if s in crosstab.columns]
crosstab = crosstab.reindex(index=row_order, columns=col_order)

plt.figure(figsize=(8, 5))
sns.heatmap(crosstab, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Proportion'})
plt.xlabel('Observed fire severity')
plt.ylabel('Predicted risk class (pre-season)')
plt.title('Predicted risk class vs. observed 2019/20 fire severity')
plt.tight_layout()
plt.savefig('outputs/validation_heatmap.png', dpi=150)
plt.show()

## 8. Interpretation notes

- This is a **qualitative** validation, not a calibrated statistical
  model. Adjacent pixels are spatially correlated (not independent
  samples), and this uses a single pre-season date — treat p-values or
  strict statistical claims with caution here.
- If the trend is weak or noisy, a good next step (mentioned in the
  README) is fitting a proper classifier (e.g. Random Forest) using the
  same NDVI/NDMI/slope/aspect features against the FESM severity labels
  as training data — that would also let you learn the layer weights
  from data instead of assigning them by hand.
- Update this notebook's markdown with your actual boxplot/heatmap
  results and a short written interpretation before referencing it in
  the README — that written interpretation is often what interviewers
  actually read.